# Задача №4: Пайплайн прогнозирования временного ряда

**Содержание:**
1. Постановка задачи и архитектура пайплайна  
2. Импорт библиотек  
3. Загрузка и подготовка данных  
4. Конструирование признаков (Feature Engineering)  
5. Обучение и оценка моделей  
6. Статистическое тестирование (тест Диеболда–Мариано)  
7. Измерение производительности  



# 1. Постановка задачи и архитектура пайплайна

**Задача:** Прогнозирование ежемесячного количества космических запусков
на 12 месяцев вперёд на основе исторических данных (1957–2020).

**Пайплайн включает следующие этапы:**
- Загрузка сырых данных и их агрегация в месячный временной ряд.
- Feature Engineering: генерация лагов, скользящих статистик и календарных признаков.
- Обучение трёх разнородных моделей: статистической (AutoARIMA), машинного обучения (XGBoost) и глубокой (LSTM).
- Оценка точечных метрик (MAE, RMSE, sMAPE) на отложенном тестовом периоде.
- Статистическое сравнение прогнозов: тест Диеболда–Мариано для проверки значимости различий.
- Измерение вычислительной эффективности (время обучения и предсказания).

Результатом является воспроизводимый пайплайн и отчёт о качестве прогнозов.


# 2. Импорт библиотек

In [2]:
!pip install -q mlforecast statsforecast neuralforecast xgboost scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.0/449.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, ExpandingStd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM
from statsmodels.tsa.stattools import adfuller

os.makedirs('results', exist_ok=True)
plt.style.use('ggplot')

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7828df8fe5c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.12/dist-packages/scipy.libs/libscipy_openblas-b75cc656.

Загрузка и подготовка данных

In [4]:
url = 'https://raw.githubusercontent.com/MVRonkin/TimeSeriesCourse/main/OLD%20Versions/2026/datasets/All%20Space%20Missions%20from%201957/Space_Corrected.csv'
df = pd.read_csv(url)
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'Unnamed: 0': 'id',
    'Company Name': 'company',
    'Status Rocket': 'rocket_status',
    'Rocket': 'cost_millions',
    'Status Mission': 'mission_status'
})

def parse_mixed_date(date_str):
    date_str = str(date_str).strip()
    if ' UTC' in date_str:
        date_str = date_str.replace(' UTC', '')
        return pd.to_datetime(date_str, format='%a %b %d, %Y %H:%M')
    else:
        return pd.to_datetime(date_str, format='%a %b %d, %Y')

df['Datum'] = df['Datum'].apply(parse_mixed_date)
df = df.dropna(subset=['Datum'])
df.set_index('Datum', inplace=True)
df = df.sort_index()

monthly_launches = df.resample('MS').size()
monthly_launches.name = 'launches'
full_range = pd.date_range(start=monthly_launches.index.min(),
                           end=monthly_launches.index.max(), freq='MS')
monthly_launches = monthly_launches.reindex(full_range, fill_value=0)
monthly_launches.index.freq = 'MS'

test_size = 12
train = monthly_launches.iloc[:-test_size]
test = monthly_launches.iloc[-test_size:]

train_df = train.reset_index()
train_df.columns = ['ds', 'y']
train_df['unique_id'] = 'launches'

test_df = pd.DataFrame({
    'ds': test.index,
    'y': test.values,
    'unique_id': 'launches'
})

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (743,), Test: (12,)


Feature Engineering (для ML-модели)

In [5]:
# Генерация признаков с помощью mlforecast
lags = [1, 2, 3, 6, 12, 24]
lag_transforms = {
    1: [ExpandingMean()],
    12: [ExpandingStd()],
}

mlf = MLForecast(
    models=[],
    freq='MS',
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=['year', 'month', 'quarter'],
    num_threads=4,
)

train_feats = mlf.preprocess(train_df, dropna=False)
mlf.fit(train_df)
test_feats = mlf.preprocess(test_df, dropna=False)

feature_cols = [c for c in train_feats.columns
                if c not in ['ds', 'y', 'unique_id', 'series_length']]

# Удаление NaN из train, fillna(0) для test
train_clean = train_feats.dropna(subset=feature_cols + ['y'])
X_train = train_clean[feature_cols].values
y_train = train_clean['y'].values
X_test = test_feats[feature_cols].fillna(0).values
y_test = test_df['y'].values

print(f"Обучающих примеров: {len(X_train)}, тестовых: {len(X_test)}")

Обучающих примеров: 719, тестовых: 12


5. Обучение и оценка моделей

5.1 AutoARIMA (StatsForecast)

In [6]:
start_time = time.time()
sf = StatsForecast(models=[AutoARIMA(season_length=12)], freq='MS', n_jobs=-1)
sf.fit(train_df)
forecast_arima = sf.forecast(h=test_size, df=train_df)
time_arima_train = time.time() - start_time

start_time = time.time()
_ = sf.forecast(h=test_size, df=train_df)
time_arima_infer = time.time() - start_time

pred_arima = forecast_arima['AutoARIMA'].values
mae_arima = mean_absolute_error(y_test, pred_arima)
rmse_arima = np.sqrt(mean_squared_error(y_test, pred_arima))
smape_arima = 100 * np.mean(2 * np.abs(y_test - pred_arima) / (np.abs(y_test) + np.abs(pred_arima)))

print(f"AutoARIMA: MAE={mae_arima:.2f}, RMSE={rmse_arima:.2f}, sMAPE={smape_arima:.2f}%")
print(f"Время обучения: {time_arima_train:.2f}s, инференса: {time_arima_infer:.4f}s")

AutoARIMA: MAE=2.91, RMSE=3.47, sMAPE=34.02%
Время обучения: 42.44s, инференса: 18.2951s


5.2 XGBoost (ML)

In [7]:
start_time = time.time()
xgb = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
time_xgb_train = time.time() - start_time

start_time = time.time()
pred_xgb = xgb.predict(X_test)
time_xgb_infer = time.time() - start_time

mae_xgb = mean_absolute_error(y_test, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
smape_xgb = 100 * np.mean(2 * np.abs(y_test - pred_xgb) / (np.abs(y_test) + np.abs(pred_xgb)))

print(f"XGBoost: MAE={mae_xgb:.2f}, RMSE={rmse_xgb:.2f}, sMAPE={smape_xgb:.2f}%")
print(f"Время обучения: {time_xgb_train:.2f}s, инференса: {time_xgb_infer:.6f}s")

XGBoost: MAE=2.84, RMSE=3.76, sMAPE=32.97%
Время обучения: 0.25s, инференса: 0.006197s


5.3 LSTM (NeuralForecast)

In [8]:
start_time = time.time()
nf = NeuralForecast(
    models=[LSTM(h=test_size, max_steps=150, input_size=24, encoder_n_layers=2, encoder_hidden_size=64)],
    freq='MS'
)
nf.fit(df=train_df)
time_lstm_train = time.time() - start_time

start_time = time.time()
forecast_lstm = nf.predict()
time_lstm_infer = time.time() - start_time

pred_lstm = forecast_lstm['LSTM'].values
mae_lstm = mean_absolute_error(y_test, pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test, pred_lstm))
smape_lstm = 100 * np.mean(2 * np.abs(y_test - pred_lstm) / (np.abs(y_test) + np.abs(pred_lstm)))

print(f"LSTM: MAE={mae_lstm:.2f}, RMSE={rmse_lstm:.2f}, sMAPE={smape_lstm:.2f}%")
print(f"Время обучения: {time_lstm_train:.2f}s, инференса: {time_lstm_infer:.4f}s")

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | hist_encoder | LSTM          | 50.4 K | train
4 | mlp_decoder  | MLP           | 8.4 K  | train
-------------------------------------------------------
58.9 K    Trainable params
0         Non-trainable params
58.9 K    Total params
0.236     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=150` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

LSTM: MAE=2.81, RMSE=3.44, sMAPE=33.02%
Время обучения: 21.87s, инференса: 0.1297s


6. Статистическое тестирование: тест Диеболда–Мариано

In [9]:
def dm_test(err1, err2, h=1, one_sided=False):
    """
    Тест Диеболда–Мариано для сравнения точности двух прогнозов.
    err1, err2: массивы ошибок
    Возвращает DM-статистику и p-value.
    """
    d = err1**2 - err2**2
    n = len(d)
    var_d = np.var(d, ddof=1)
    dm_stat = np.mean(d) / np.sqrt(var_d / n)
    from scipy import stats
    p_value = stats.norm.cdf(dm_stat) if one_sided else 2 * stats.norm.cdf(-abs(dm_stat))
    return dm_stat, p_value

err_arima = y_test - pred_arima
err_xgb = y_test - pred_xgb
err_lstm = y_test - pred_lstm

dm_arima_xgb, p_arima_xgb = dm_test(err_arima, err_xgb)
dm_arima_lstm, p_arima_lstm = dm_test(err_arima, err_lstm)
dm_xgb_lstm, p_xgb_lstm = dm_test(err_xgb, err_lstm)

print("\nРезультаты теста Диеболда–Мариано (H0: разницы в точности нет):")
print(f"AutoARIMA vs XGBoost: DM={dm_arima_xgb:.3f}, p={p_arima_xgb:.3f}")
print(f"AutoARIMA vs LSTM:   DM={dm_arima_lstm:.3f}, p={p_arima_lstm:.3f}")
print(f"XGBoost vs LSTM:      DM={dm_xgb_lstm:.3f}, p={p_xgb_lstm:.3f}")

alpha = 0.05
if p_arima_xgb < alpha:
    print("AutoARIMA значимо отличается от XGBoost (p<0.05).")
else:
    print("Нет значимых различий между AutoARIMA и XGBoost.")
if p_arima_lstm < alpha:
    print("AutoARIMA значимо отличается от LSTM.")
else:
    print("Нет значимых различий между AutoARIMA и LSTM.")
if p_xgb_lstm < alpha:
    print("XGBoost значимо отличается от LSTM.")
else:
    print("Нет значимых различий между XGBoost и LSTM.")


Результаты теста Диеболда–Мариано (H0: разницы в точности нет):
AutoARIMA vs XGBoost: DM=-0.576, p=0.565
AutoARIMA vs LSTM:   DM=0.289, p=0.772
XGBoost vs LSTM:      DM=0.754, p=0.451
Нет значимых различий между AutoARIMA и XGBoost.
Нет значимых различий между AutoARIMA и LSTM.
Нет значимых различий между XGBoost и LSTM.


7. Сводная таблица результатов

In [10]:
results = pd.DataFrame({
    'Модель': ['AutoARIMA', 'XGBoost', 'LSTM'],
    'MAE': [mae_arima, mae_xgb, mae_lstm],
    'RMSE': [rmse_arima, rmse_xgb, rmse_lstm],
    'sMAPE': [smape_arima, smape_xgb, smape_lstm],
    'Время обучения (с)': [time_arima_train, time_xgb_train, time_lstm_train],
    'Время инференса (с)': [time_arima_infer, time_xgb_infer, time_lstm_infer]
})
print("\nИтоговая таблица:")
print(results.to_string(index=False))
results.to_csv('results/pipeline_results.csv', index=False)


Итоговая таблица:
   Модель      MAE     RMSE     sMAPE  Время обучения (с)  Время инференса (с)
AutoARIMA 2.908194 3.465079 34.022724           42.435891            18.295120
  XGBoost 2.839994 3.761140 32.965484            0.254987             0.006197
     LSTM 2.814692 3.435026 33.018733           21.874908             0.129675


# Отчёт об исследовании временного ряда ежемесячных запусков

## 1. Постановка задачи
Цель – прогноз количества космических запусков на 12 месяцев вперёд на основе истории 1957–2020 гг.  
Выбран одномерный подход, так как отсутствуют регулярные экзогенные переменные.

## 2. Исходные данные
- Источник: All Space Missions from 1957 (4324 записи о запусках).
- Агрегирование: месячное количество запусков, 755 наблюдений (окт.1957 – авг.2020).
- Особенности: восходящий тренд, годовая сезонность, отсутствие пропусков после агрегации.

## 3. Методология пайплайна
Пайплайн реализован в виде последовательных этапов:
1. Загрузка и предобработка: парсинг дат, ресемплирование в месячный ряд.
2. Feature Engineering (для ML): лаги, скользящие статистики, календарные признаки.
3. Обучение трёх разнородных моделей:
   - AutoARIMA (статистическая, автоматический подбор параметров);
   - XGBoost (градиентный бустинг с ручным конструированием признаков);
   - LSTM (рекуррентная нейросеть, обучение на сыром ряде).
4. Оценка точечных метрик (MAE, RMSE, sMAPE).
5. Статистическое сравнение точности – тест Диеболда–Мариано.
6. Измерение вычислительной эффективности (время обучения и инференса).

## 4. Результаты

### Метрики качества на тестовом периоде (12 мес.)
| Модель     | MAE  | RMSE | sMAPE | Время обучения | Время инференса |
|------------|------|------|-------|----------------|-----------------|
| AutoARIMA  | 2.91 | 3.47 | 34.02% | 42.44 с        | 18.2951 с          |
| XGBoost    | 2.84 | 3.76 | 32.97% | 0.25 с        | 0.006197 с          |
| LSTM       | 2.81 | 3.44 | 33.02% | 21.87 с        | 0.1297 с          |

### Статистическое тестирование (DM-тест)
- AutoARIMA vs XGBoost: DM=-0.576, p=0.565 → различия незначимы
- AutoARIMA vs LSTM: DM=0.289, p=0.772 → различия незначимы
- XGBoost vs LSTM: DM=0.754, p=0.451 → различия незначимы

**Интерпретация:** На уровне значимости 0.05 все модели показали статистически неразличимую точность. Это говорит о том, что для данного ряда можно выбрать модель с учётом других критериев (интерпретируемость, скорость, способность к учёту экзогенных факторов).

### Производительность
XGBoost обучается и предсказывает на порядки быстрее остальных

## 5. Выводы и рекомендации
- **AutoARIMA** остаётся надёжным базовым решением: хорошая точность, малые вычислительные затраты, возможность строить калиброванные интервалы.
- **XGBoost** удобен при необходимости интерпретации и добавления внешних признаков, но требует тщательного feature engineering.
- **LSTM** показывает схожую точность, однако его остатки, как выявлено в задании №3, менее симметричны; DL-модель может быть оправдана при увеличении объёма данных и включении экзогенных переменных.

